## Робота з базовими функціями граф-орієнтованої БД на прикладі Neo4j 
___

Завдання:
Змоделювати предметну область онлайн-магазину:
- Є: Items(id, name, price), Customers(id, name), Orders(id, date)
- Customer може мати (bought) багато Orders
- Item може входити (contains) в декілька Orders
- Customer може переглядати (view), але при цьому не купувати Items 

Написати наступні види запитів:


In [ ]:
#import importlib
#import conect_to_neo4j 
#importlib.reload(conect_to_neo4j)
#print(dir(conect_to_neo4j))

from conect_to_neo4j import execute_query,create_schema,driver
create_schema()

1. Знайти Items які входять в конкретний Order (за Order id) 


In [ ]:
records = execute_query(
        """
        MATCH (o:Orders {id: 2})-[:CONTAINS]->(i:Items)
        RETURN i.name AS item_name
        """
    )

print("Items in Order:")
for record in records:
    print(record["item_name"])

2. Підрахувати вартість конкретного Order


In [ ]:
records = execute_query(
        """
        MATCH (o:Orders {id: 1})-[:CONTAINS]->(i:Items)
        RETURN SUM(i.price) AS total_price
        """
    )

print("Total cost of Order:", records[0]["total_price"])

3. Знайти всі Orders конкретного Customer


In [ ]:
records = execute_query(
        """
        MATCH (c:Customer {id: 2})-[:BOUGHT]->(o:Orders) 
        RETURN o.id as cust_ord
        """
    )

print("Orders of customer :")
for record in records:
    print(record["cust_ord"])


4. Знайти всі Items куплені конкретним Customer (через його Orders)


In [ ]:
records = execute_query(
    """
    MATCH (c:Customer {id: 1})-[:BOUGHT]->(:Orders)-[:CONTAINS]->(i:Items)
    RETURN DISTINCT i.name AS item_name
    """
    )

print("Items bought by Customer:")
for record in records:
    print(record["item_name"])


5. Знайти загальну кількість Items куплені конкретним Customer (через його Order)


In [ ]:
records = execute_query(
    """
    MATCH (c:Customer {id: 2})-[:BOUGHT]->(:Orders)-[:CONTAINS]->(i:Items)
    RETURN COUNT(i) AS total_items
    """
    )

print("Total items bought:", records[0]["total_items"])


6. Знайти для Customer на яку загальну суму він придбав товарів (через його Order)

In [ ]:
records = execute_query(
    """
    MATCH (c:Customer {id: 2})-[:BOUGHT]->(:Orders)-[:CONTAINS]->(i:Items) 
    RETURN SUM(i.price) AS total_spent
    """
    )

print("Total amount spent:", records[0]["total_spent"])


7. Знайті скільки разів кожен товар був придбаний, відсортувати за цим значенням

In [ ]:
records = execute_query(
    """
    MATCH (:Orders)-[:CONTAINS]->(i:Items) RETURN i.name AS item_name, 
    COUNT(*) AS times_bought ORDER BY times_bought DESC
    """
    )

print("Item purchase count:")
for record in records:
    print(f"{record['item_name']}: {record['times_bought']}")


8. Знайти всі Items переглянуті (view) конкретним Customer


In [ ]:
records = execute_query(
    """
    MATCH (c:Customer {id: 3})-[:VIEWED]->(i:Items)
    RETURN i.name AS item_name
    """
    )

print("Items viewed by Customer:")
for record in records:
    print(record["item_name"])


9. Знайти інші Items що купувались разом з конкретним Item (тобто всі Items що входять до Order-s разом з даними Item)

In [ ]:
records = execute_query(
    """
    MATCH (o:Orders)-[:CONTAINS]->(i:Items {id: 2})
    MATCH (o)-[:CONTAINS]->(other:Items)
    WHERE other.id <> 2
    RETURN DISTINCT other.name AS item_name
    """
    )

print("Items bought together with the given Item:")
for record in records:
    print(record["item_name"])


10. Знайти Customers які купили даний конкретний Item


In [ ]:
records = execute_query(
    """
    MATCH (c:Customer)-[:BOUGHT]->(o:Orders)-[:CONTAINS]->(i:Items {id: 2})
    RETURN DISTINCT c.name AS customer_name
    """
    )

print("Customers who bought the given Item:")
for record in records:
    print(record["customer_name"])


11. Знайти для певного Customer(а) товари, які він переглядав, але не купив


In [ ]:
records = execute_query(
    """
    MATCH (c:Customer {id: 3})-[:VIEWED]->(i:Items)
    WHERE NOT EXISTS {
        MATCH (c)-[:BOUGHT]->(:Orders)-[:CONTAINS]->(i)
    }
    RETURN i.name AS item_name
    """
    )

print("Items viewed but not bought by Customer:")
for record in records:
    print(record["item_name"])



Закриваємо сесію

In [ ]:
driver.close()